# 00 Environment, paths and the filter in one picture

In [ ]:
# find the repo root (works from notebooks/<group>/, from the repo root, on Kaggle and on Colab) and set up the folders
import os, sys, shutil
from pathlib import Path
KAGGLE_REPO = Path("/kaggle/input/datasets/sakhiurrahman/fedupds/FedUP_Reproduction")   # your Kaggle dataset (edit if you move it)
def _find_repo():
    for p in [Path.cwd(), *Path.cwd().parents, Path("/kaggle/working/FedUP_Reproduction"), Path("/content/FedUP_Reproduction"), KAGGLE_REPO]:
        if (p / "fedup_repro").exists():
            return p
    kin = Path("/kaggle/input")
    if kin.exists():                                    # any other layout of the input dataset, up to 6 levels deep
        for dp, dn, _ in os.walk(kin):
            if "fedup_repro" in dn:
                return Path(dp)
            if len(Path(dp).relative_to(kin).parts) > 6:
                dn[:] = []
    return None
ROOT = _find_repo()
assert ROOT is not None, "FedUP_Reproduction folder not found: upload it (Kaggle: as a Dataset) or cd into it"
if str(ROOT).startswith("/kaggle/input"):              # input datasets are read-only: work on a copy of the code ...
    SRC, ROOT = ROOT, Path("/kaggle/working") / ROOT.name
    if not ROOT.exists():
        shutil.copytree(SRC, ROOT, ignore=lambda d, names: [n for n in names if n == "raw" and Path(d).name == "data"])
    os.environ["FEDUP_BUNDLED_RAW"] = str(SRC / "data" / "raw")      # ... and LINK (not copy) the datasets that ship with it
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
from fedup_repro import colab
PERSIST_ROOT = None        # Colab + Drive: "/content/drive/MyDrive/fedup_store" keeps data/ and artifacts/ across sessions (mount Drive first)
print("platform:", colab.setup_dirs(PERSIST_ROOT), "| repo:", ROOT)

In [ ]:
import torch
from fedup_repro import colab, paths
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(), "| GPUs", torch.cuda.device_count())
print(colab.gpu_summary())
for n in ["DATA_DIR", "PROCESSED_DIR", "ARTIFACTS_DIR", "CKPT_DIR", "FILTER_DIR", "RESULTS_DIR"]:
    print(f"{n:14s}", getattr(paths, n))
colab.status()

## The model with and without the filter
The filter is a 2-layer bottleneck between the (frozen) extractor and the classifier. Dropping it restores the original model.

In [ ]:
from fedup_repro.config import make_config
from fedup_repro.models import build_model, Filter, FilteredModel
cfg = make_config("MNIST-LeNet5", "client")
m = build_model(cfg); f = Filter(m.feat_dim, cfg["out_dim"])
print("feature dim", m.feat_dim, "| filter parameters", sum(p.numel() for p in f.parameters()), "| filter size KB", f.nbytes()/1024)
x = torch.randn(4, 1, 28, 28)
print("original:", m(x).shape, "| with filter:", FilteredModel(m, f)(x).shape)